In [5]:
# [사전 준비] (코랩에서 최초 1회 실행)
!pip install konlpy

import pandas as pd
import re
import numpy as np
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# 0. 데이터 불러오기 및 통합
# ==========================================
# 파일명 오류 방지를 위해 업로드하신 파일명을 그대로 사용합니다.
past_df = pd.read_excel("/content/26_1_데이터사이언스개론_팀플_과거전시.xlsx")
current_df = pd.read_excel("/content/26_1_데이터사이언스개론_팀플_현재전시.xlsx")

past_df['type'] = 'past'
current_df['type'] = 'current'

df = pd.concat([past_df, current_df], ignore_index=True)
df = df.dropna(subset=['description']).copy()

# ==========================================
# 1~3단계: 데이터 정제 및 형태소 분석
# ==========================================
okt = Okt()
stopwords = [
    '전시', '작품', '작가', '미술관', '국립', '현대', '안내', '개최', '관람', '예술', '장소', '일시',
    '진행', '소장품', '소개', '제공', '이후', '중심', '과정', '의미', '스스로', '주요', '최근', '가지'
]

def clean_and_tokenize(text):
    text = re.sub('<[^>]*>', ' ', str(text))
    text = re.sub('[^가-힣\s]', ' ', text)

    words = okt.pos(text, stem=True)
    keywords = [word for word, pos in words if pos == 'Noun' and len(word) > 1 and word not in stopwords]
    return ' '.join(keywords)

df['tokenized_text'] = df['description'].apply(clean_and_tokenize)

# ==========================================
# 4단계: TF-IDF 기반 고유 핵심 태그(top_tags) 추출
# ==========================================
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['tokenized_text'])
feature_names = np.array(tfidf.get_feature_names_out())

def get_tfidf_top_tags(row_idx, top_n=5):
    row_scores = tfidf_matrix[row_idx].toarray()[0]
    top_indices = row_scores.argsort()[-top_n:][::-1]
    top_words = feature_names[top_indices]
    return ', '.join(top_words)

df['top_tags'] = [get_tfidf_top_tags(i) for i in range(len(df))]

# ==========================================
# 5단계: 4대 장르 스코어링 및 상위 2개 제한 추출 (고도화 로직)
# ==========================================
# 세부 장르들을 4가지 대분류(Macro-category)로 완벽히 통합했습니다.
taxonomy_dict_broad = {
    '매체 장르': ['회화', '한국화', '판화', '사진', '드로잉', '평면', '캔버스', '글씨', '서예', '한지',
              '조각', '설치', '공예', '건축', '입체', '오브제', '공간', '물성', '양감', '조형물',
              '미디어', '미디어아트', '영상', '사운드', '디지털', '인터랙티브', '애니메이션', '영화', '스크린'],

    '시대/성격 장르': ['유물', '보물', '역사', '민속', '고고학', '조선', '고대', '가야', '문화재', '전통', '불교',
                '근대', '시대', '사건', '기록', '독립', '전쟁', '해방', '역사적',
                '현대미술', '동시대', '실험', '개념', '추상', '새롭다', '아방가르드'],

    '기획 목적 장르': ['개인전', '회고전', '일생', '생애', '세계관', '거장', '조명', '선생', '유작',
                '특별전', '기획전', '주제', '기획', '공동기획', '교류전',
                '기증', '소장품', '컬렉션', '수장고', '수집', '기증품'],

    '경험 장르': ['어린이', '가족', '체험', '참여', '놀이', '워크숍', '교육', '체험형',
              '아카이브', '기록물', '도면', '편지', '문헌', '자료', '도록', '인터뷰', '구술',
              '몰입', '감상', '분위기', '풍경', '빛', '색채', '몽환', '고요함', '오감', '시각적']
}

def map_top_2_genres(keyword_string, top_n=2):
    words = keyword_string.split()
    genre_scores = {'매체 장르': 0, '시대/성격 장르': 0, '기획 목적 장르': 0, '경험 장르': 0}

    # 각 대분류별로 단어 출현 빈도를 점수화
    for word in words:
        for broad_genre, keywords in taxonomy_dict_broad.items():
            if word in keywords:
                genre_scores[broad_genre] += 1

    # 0점인 장르 필터링
    valid_scores = {k: v for k, v in genre_scores.items() if v > 0}

    if not valid_scores:
        return '기타/미분류'

    # 점수가 높은 순으로 정렬하여 상위 2개만 추출
    sorted_genres = sorted(valid_scores.items(), key=lambda item: item[1], reverse=True)
    top_genres = [genre for genre, score in sorted_genres[:top_n]]

    return ', '.join(top_genres)

df['genres'] = df['tokenized_text'].apply(map_top_2_genres)

# ==========================================
# 6. 결과물 분리 및 엑셀 저장
# ==========================================
final_df = df[['title', 'cntc_instt_nm', 'period', 'top_tags', 'genres', 'type']]

final_past = final_df[final_df['type'] == 'past'].drop(columns=['type'])
final_current = final_df[final_df['type'] == 'current'].drop(columns=['type'])

final_past.to_excel('최종_4대장르_과거전시.xlsx', index=False)
final_current.to_excel('최종_4대장르_현재전시.xlsx', index=False)

print("✨ TF-IDF 가중치 추출 + 4대 장르 스코어링(최대 2개 제한) 완료! ✨")
print("📁 '최종_4대장르_과거전시.xlsx', '최종_4대장르_현재전시.xlsx' 로 각각 저장되었습니다.\n")

# 데이터 미리보기 확인
print(final_current[['title', 'top_tags', 'genres']].head(5))

<>:34: SyntaxWarning: invalid escape sequence '\s'
<>:34: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2981/3495397895.py:34: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('[^가-힣\s]', ' ', text)


✨ TF-IDF 가중치 추출 + 4대 장르 스코어링(최대 2개 제한) 완료! ✨
📁 '최종_4대장르_과거전시.xlsx', '최종_4대장르_현재전시.xlsx' 로 각각 저장되었습니다.

                       title             top_tags              genres
95    로드 무비: 1945년 이후 한·일 미술  교류, 무비, 로드, 예술가, 대칭  시대/성격 장르, 기획 목적 장르
96                   오~감각미술관   더욱, 감각, 만날, 세계, 걸음        경험 장르, 매체 장르
97    방혜자 - 천지에 마음의 빛 뿌리며 간다  혜자, 프랑스, 양분, 기별, 원천     시대/성격 장르, 매체 장르
98  MMCA 과천 상설전 «한국근현대미술 II»  미술, 한국, 흐름, 미술사, 여성  시대/성격 장르, 기획 목적 장르
99   MMCA 과천 상설전 «한국근현대미술 I»   조선, 시기, 미술, 설전, 과천     시대/성격 장르, 매체 장르
